In [69]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [70]:
import torch
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tqdm 

import what_where as ww
from utils import get_day_orientations, add_correct, add_change_bins


In [71]:
cfg = ww.utils.load_config("config_orientation_change_detection")
results_dir = ww.utils.RESULTS_DIR / cfg.experiment.name
results_dir.mkdir(parents=True, exist_ok=True)

cfg.train.instance = 0

results_path = results_dir / f"{cfg.experiment.name}_results_data.parquet"
activations_path = results_dir / f"{cfg.experiment.name}_activations.npy"


In [72]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# loading the latest checkpoint
checkpoint_path = ww.utils.get_checkpoint_path(cfg)
print("checkpoint file: ", checkpoint_path)

if checkpoint_path is None:
    raise ValueError("No checkpoint found for config")
checkpoint = torch.load(checkpoint_path, weights_only=False)

model = ww.model.Model(cfg)

dataset = ww.utils.get_datasets(cfg)["valid"]
dataset.attend_valid_prob = 0.8

# replicating from the paper a set of fixed starting orientations with many repetitions
day_orientations = get_day_orientations(cfg)
dataset.set_day_orientations(day_orientations)

dataloader = DataLoader(dataset, batch_size=cfg.train.dataloader.batch_size, shuffle=True)

# loading model weights
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()
None

checkpoint dir:  /home/eivinas/dev/what-where/checkpoints/orientation_change_detection/orientation_change_detection/ean_full/instance0
checkpoint:  /home/eivinas/dev/what-where/checkpoints/orientation_change_detection/orientation_change_detection/ean_full/instance0/checkpoint_050.pth 

checkpoint file:  /home/eivinas/dev/what-where/checkpoints/orientation_change_detection/orientation_change_detection/ean_full/instance0/checkpoint_050.pth


In [73]:

n_dataset = len(dataset)
n_passes = cfg.model.n_passes
last_pass = n_passes - 1    

activations = {}
for conv_layer in cfg.experiment.conv_layers:
    activations[conv_layer] = {}
    for side in ["left", "right"]:
        n_channels = 64 if conv_layer == "conv1" else 256
        activations[conv_layer][side] = torch.zeros((n_dataset, cfg.experiment.n_reps, len(cfg.experiment.energy_costs), n_passes, n_channels))

def get_orientation_change_detection_data(model, dataloader):
    data = {
        "day" : [],
        "orientation_left" : [],
        "orientation_right" : [],
        "attend_left" : [],
        "attend_valid" : [],
        "change_side":  [],
        "change": [],
        "change_t": [],
        "target" : [],
        "target_pred" : [],
        "energy_cost" : [],
        "energy_use" : [],
        "t" : [],
        "repetition" : [],
        "trial" : [],
    }


    for i, (images, labels,  meta_info) in enumerate(tqdm.tqdm(dataloader, desc="Processing batches")):
        images = images.to(device)
        target = labels["what"]

        start_index = i * cfg.train.dataloader.batch_size
        end_index = start_index + images.shape[0]

        for k, energy_cost in enumerate(cfg.experiment.energy_costs):
            log_energy_cost = torch.tensor(energy_cost).to(device).repeat(images.shape[0], 1)

            for rep in range(cfg.experiment.n_reps):
                with torch.no_grad():
                    out_full = model(images, log_energy_cost, n_passes, noise_anneal=1.0)

                    for t in range(n_passes):
                        for key in data.keys():
                            if key in meta_info:
                                data[key].extend(meta_info[key].cpu().numpy())

                        # pass energy
                        energy_use = torch.zeros((images.shape[0]), device=device)
                        energy_use_t = ww.utils.get_energy_use(cfg, out_full, t, device)
                        for key in energy_use_t.keys():
                            energy_use += energy_use_t[key]

                        data["energy_cost"].extend([energy_cost] * images.shape[0])
                        data["energy_use"].extend(energy_use.cpu().numpy())
                        data["t"].extend([t] * images.shape[0])
                        data["trial"].extend(range(start_index, end_index))
                        data["repetition"].extend([rep] * images.shape[0])

                        data["target"].extend(target[:,t].cpu().numpy())
                        target_pred = torch.argmax(out_full[t]["prediction"]["what"], dim=-1).cpu()
                        data["target_pred"].extend(target_pred.numpy())

                        # activations
                        for side in ["left", "right"]:
                            for conv_layer in cfg.experiment.conv_layers:

                                pos = list(cfg.dataset.center_left) if side == "left" else list(cfg.dataset.center_right)
                                ratio = 64 / 32 if conv_layer == "conv1" else 64 / 8
                                unit_index = [int(pos[0] / ratio), int(pos[1] / ratio)]  # convert position of the grating to layer coordinates

                                activation = out_full[t]["activations"][conv_layer][:, :, unit_index[0], unit_index[1]]
                                activations[conv_layer][side][start_index:end_index, rep, k, t] = activation.cpu()

    print("creating the dataframe")
    df_full = pd.DataFrame(data)
    print("dataframe created with shape:", df_full.shape)

    return df_full, activations



if results_path.exists() and not cfg.experiment.redo_extraction:
    print("Loading existing orientation change detection data...")
    df_full = pd.read_parquet(results_path)
    print("orientation change detection data loaded with shape:", df_full.shape)
else:
    print("Processing orientation change detection data...")
    df_full, activations = get_orientation_change_detection_data(model, dataloader)
    df_full.to_parquet(results_path, index=False)

    print("saving activations...")
    activations_np = {}
    for conv_layer in activations.keys():
        activations_np[conv_layer] = {}
        for side in activations[conv_layer].keys():
            activations_np[conv_layer][side] = activations[conv_layer][side].cpu().numpy()
    np.save(activations_path, activations_np)

Processing orientation change detection data...


Processing batches: 100%|██████████| 16/16 [00:21<00:00,  1.36s/it]


creating the dataframe
dataframe created with shape: (160000, 15)
saving activations...


In [74]:
df_full

,day,orientation_left,orientation_right,attend_left,attend_valid,change_side,change,change_t,target,target_pred,energy_cost,energy_use,t,repetition,trial
0,40,2.711519,1.958149,False,True,1,-3.797198,3,1,1,-10.0,5644.437012,0,0,0
1,14,1.861125,0.145928,False,True,1,-43.307950,3,1,1,-10.0,5675.843750,0,0,1
2,1,2.299627,1.880741,False,True,1,46.262792,2,1,1,-10.0,5592.806641,0,0,2
3,29,0.142086,1.022055,True,False,1,-13.400499,1,1,1,-10.0,5669.344238,0,0,3
4,22,0.812981,2.081375,True,True,0,-37.524080,1,1,1,-10.0,5581.593750,0,0,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159995,16,0.204366,2.981012,False,True,1,65.589606,3,2,2,-10.0,2233.038818,3,19,1995
159996,1,2.299627,1.880741,False,True,1,17.780114,1,1,1,-10.0,1940.795532,3,19,1996
159997,49,0.079857,0.338951,True,True,0,20.105440,3,0,0,-10.0,2029.234985,3,19,1997
159998,13,0.627294,1.615515,False,True,1,10.068486,2,1,1,-10.0,2432.096191,3,19,1998


In [75]:
df_full = add_correct(df_full)
df_full = add_change_bins(df_full)

In [76]:
df_full["change_bin"].unique()

[5.462225, 39.310852, 10.545906, 20.360956, 75.897373, 1.465349, 2.829146]
Categories (7, float64): [1.465349 < 2.829146 < 5.462225 < 10.545906 < 20.360956 < 39.310852 < 75.897373]

In [77]:
df_full[df_full["attend_valid"] == False]["correct"].mean()
df_full

,day,orientation_left,orientation_right,attend_left,attend_valid,change_side,change,change_t,target,target_pred,energy_cost,energy_use,t,repetition,trial,correct,change_bin
0,40,2.711519,1.958149,False,True,1,-3.797198,3,1,1,-10.0,5644.437012,0,0,0,True,5.462225
1,14,1.861125,0.145928,False,True,1,-43.307950,3,1,1,-10.0,5675.843750,0,0,1,True,39.310852
2,1,2.299627,1.880741,False,True,1,46.262792,2,1,1,-10.0,5592.806641,0,0,2,True,39.310852
3,29,0.142086,1.022055,True,False,1,-13.400499,1,1,1,-10.0,5669.344238,0,0,3,True,10.545906
4,22,0.812981,2.081375,True,True,0,-37.524080,1,1,1,-10.0,5581.593750,0,0,4,True,39.310852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
159995,16,0.204366,2.981012,False,True,1,65.589606,3,2,2,-10.0,2233.038818,3,19,1995,True,75.897373
159996,1,2.299627,1.880741,False,True,1,17.780114,1,1,1,-10.0,1940.795532,3,19,1996,True,20.360956
159997,49,0.079857,0.338951,True,True,0,20.105440,3,0,0,-10.0,2029.234985,3,19,1997,True,20.360956
159998,13,0.627294,1.615515,False,True,1,10.068486,2,1,1,-10.0,2432.096191,3,19,1998,True,10.545906
